In [1]:
! pip install duckdb

In [3]:
import duckdb

try:
    con.close() 
except NameError:
    pass

con = duckdb.connect("../dataset/goodreads.db") 
con.sql("SET memory_limit='4GB'")
con.sql("SET temp_directory='duckdb_tmp'")

In [5]:
con.sql("""
    CREATE OR REPLACE TABLE interactions_raw AS
    SELECT user_id, book_id AS book_id_csv, is_read, rating, is_reviewed
    FROM read_csv_auto('../dataset_original/goodreads_interactions.csv')
""")

con.sql("SELECT COUNT(*) AS n_interactions FROM interactions_raw").show()

┌────────────────┐
│ n_interactions │
│     int64      │
├────────────────┤
│      228648342 │
└────────────────┘



In [7]:
#look at rating distribution
con.sql("""
    SELECT rating, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM interactions_raw
    GROUP BY rating
    ORDER BY rating
""").show()

┌────────┬───────────┬────────────┐
│ rating │     n     │ percentage │
│ int64  │   int64   │   double   │
├────────┼───────────┼────────────┤
│      0 │ 124096793 │      54.27 │
│      1 │   2050529 │        0.9 │
│      2 │   6189946 │       2.71 │
│      3 │  23307457 │      10.19 │
│      4 │  37497451 │       16.4 │
│      5 │  35506166 │      15.53 │
└────────┴───────────┴────────────┘



In [8]:
# check for duplicate user book pairs
con.sql("SET memory_limit='8GB'")
con.sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT (user_id, book_id_csv)) AS n_unique_pairs
    FROM interactions_raw
""").show()

┌───────────┬────────────────┐
│  n_rows   │ n_unique_pairs │
│   int64   │     int64      │
├───────────┼────────────────┤
│ 228648342 │      228648342 │
└───────────┴────────────────┘



In [9]:
con.sql("""
    CREATE OR REPLACE TABLE interactions AS
    SELECT
        i.user_id,
        m.book_id AS book_id,
        i.is_read,
        i.rating,
        i.is_reviewed
    FROM interactions_raw i
    JOIN read_csv_auto('../dataset_original/book_id_map.csv') m
      ON i.book_id_csv = m.book_id_csv
    JOIN books_clean b
      ON m.book_id = b.book_id
""")

con.sql("SELECT COUNT(*) AS n_interactions FROM interactions").show()

┌────────────────┐
│ n_interactions │
│     int64      │
├────────────────┤
│      228648316 │
└────────────────┘



In [11]:
#check how many rows were dropped
con.sql("""
    SELECT
        (SELECT COUNT(*) FROM interactions_raw) AS before_rows,
        (SELECT COUNT(*) FROM interactions) AS after_rows
""").show()

┌─────────────┬────────────┐
│ before_rows │ after_rows │
│    int64    │   int64    │
├─────────────┼────────────┤
│   228648342 │  228648316 │
└─────────────┴────────────┘



In [12]:
con.close() 